# FI-2010 DeepLOB to HK 7709 transfer experiment

## tl;dr

The repository's FI-2010 checkpoint retains **75.35%** accuracy on its original FI-2010 test split but reaches only **30.42% accuracy** and **23.10% macro F1** on the held-out portion of HK 7709 for 9 July 2026. The zero-shot model predicts the stationary class for 78.9% of samples. This is evidence of substantial cross-market feature/distribution shift, not a checkpoint-loading failure.

## Context & Methods

The source CSV is an order-level event stream. Message types 30, 31, and 32 are applied as add, quantity-change, and delete operations. Type 50 trade prints are not separately deducted because the book update messages carry the corresponding state change. Rows sharing a `SendTime` are processed atomically before a snapshot is considered.

### Key Assumptions

- Side 0 is bid and side 1 is ask, verified from the positive spread after reconstruction.
- One state is sampled after every 10 complete book-update timestamp groups.
- Inputs use FI-2010 decimal-precision scaling: price and size are divided by 100,000.
- Labels use DeepLOB Equation (3), a 10-snapshot future mean (approximately 100 book-update groups).
- The first 20% of states only calibrates the stationary threshold to the FI-2010 test stationary share; model weights are untouched. Windows never cross the lunch break.

In [1]:
from pathlib import Path
import json
import numpy as np

project_root = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
result_path = project_root / 'output/results/fi2010_to_7709_zero_shot.json'
snapshot_path = project_root / 'data/processed/hk07709_2026-07-09_lob.npz'
result = json.loads(result_path.read_text())
snapshot = np.load(snapshot_path, allow_pickle=False)
metadata = json.loads(str(snapshot['metadata']))
print('Result:', result_path)
print('Snapshots:', snapshot_path)

Result: /Users/jiaqianxing/Documents/GitHub/DeepLOB-Deep-Convolutional-Neural-Networks-for-Limit-Order-Books/output/results/fi2010_to_7709_zero_shot.json
Snapshots: /Users/jiaqianxing/Documents/GitHub/DeepLOB-Deep-Convolutional-Neural-Networks-for-Limit-Order-Books/data/processed/hk07709_2026-07-09_lob.npz


## Data

The following profile is read from the generated artifact rather than copied from terminal output.

In [2]:
profile = {
    'raw_rows': sum(metadata['message_counts'].values()),
    'snapshots': metadata['snapshot_count'],
    'morning_snapshots': metadata['session_counts']['1'],
    'afternoon_snapshots': metadata['session_counts']['2'],
    'invalid_rows': metadata['invalid_rows'],
    'out_of_order_timestamps': metadata['out_of_order_send_times'],
    'crossed_or_locked_candidates_skipped': metadata['book_errors'].get('crossed_or_locked_snapshot', 0),
}
profile

{'raw_rows': 5494189,
 'snapshots': 160676,
 'morning_snapshots': 101060,
 'afternoon_snapshots': 59616,
 'invalid_rows': 0,
 'out_of_order_timestamps': 0,
 'crossed_or_locked_candidates_skipped': 60}

## Results

Accuracy is shown beside balanced accuracy and macro F1 because the class prediction distribution is strongly skewed.

In [3]:
model = result['model']
baseline = result['majority_baseline']
summary = {
    'FI-2010 checkpoint on FI-2010': {'accuracy': 0.7534985088, 'macro_f1': 0.7533},
    'FI-2010 checkpoint on HK 7709': {
        'accuracy': model['accuracy'],
        'balanced_accuracy': model['balanced_accuracy'],
        'macro_f1': model['macro_f1'],
    },
    'HK 7709 majority baseline': {
        'accuracy': baseline['accuracy'],
        'balanced_accuracy': baseline['balanced_accuracy'],
        'macro_f1': baseline['macro_f1'],
    },
}
summary

{'FI-2010 checkpoint on FI-2010': {'accuracy': 0.7534985088,
  'macro_f1': 0.7533},
 'FI-2010 checkpoint on HK 7709': {'accuracy': 0.3042157885720515,
  'balanced_accuracy': 0.31837782787406954,
  'macro_f1': 0.2309932483714868},
 'HK 7709 majority baseline': {'accuracy': 0.34555605737334727,
  'balanced_accuracy': 0.3333333333333333,
  'macro_f1': 0.17120855251862957}}

In [4]:
prediction_share = np.array(model['prediction_counts']) / model['samples']
label_share = np.array(result['preprocessing']['evaluation_label_counts']) / model['samples']
{
    'class_names': ['down', 'stationary', 'up'],
    'actual_share': label_share.round(4).tolist(),
    'prediction_share': prediction_share.round(4).tolist(),
    'confusion_matrix': model['confusion_matrix'],
}

{'class_names': ['down', 'stationary', 'up'],
 'actual_share': [0.3422, 0.3122, 0.3456],
 'prediction_share': [0.194, 0.7889, 0.0171],
 'confusion_matrix': [[7991, 35276, 684],
  [8993, 30335, 766],
  [7931, 35704, 742]]}

## Takeaways

- The zero-shot checkpoint is not usable on 7709 as-is: its 30.42% accuracy is below the 34.56% majority-class accuracy.
- Macro F1 is above the majority baseline because the model occasionally identifies down moves, but up-move recall is only 1.67%.
- The dominant failure is domain shift: the model predicts stationary for 78.9% of test samples even though only 31.2% are labelled stationary.
- A genuine reproduction of the paper's second experiment requires the authors' historical LSE order-book data or an equivalent multi-day LOB source. That dataset is not included in the repository. The single 7709 day cannot support the paper's previous-five-day normalization or an independent multi-day test.